In [1]:
import pandas as pd
import numpy as np
import os
import re

# ============================================================
# CONFIGURATION
# ============================================================

FUEL_FILE = (
    "/content/vahan-vehicle-registrations-by-fuel-type-nmiqeb.csv"
)

CATEGORY_FILE = (
    "/content/vahan-vehicle-registrations-by-vehicle-category-71m55i.csv"
)

MAKER_FILE = (
    "/content/vahan-vehicle-registrations-by-maker.csv"
)

OUTPUT_DIR = "data/processed"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_columns(df):

    df.columns = [
        re.sub(
            r"[^a-z0-9]+",
            "_",
            str(col).strip().lower()
        ).strip("_")
        for col in df.columns
    ]

    return df


def load_vahan(file):

    print("\nLoading:")
    print(file)

    df = pd.read_csv(file)

    df = clean_columns(df)

    print("Shape:", df.shape)

    print("Columns:")
    print(df.columns.tolist())

    return df


def clean_common_columns(df):

    # Date
    if "date" in df.columns:

        df["date"] = pd.to_datetime(
            df["date"],
            errors="coerce"
        )

    # Registration count
    if "registrations" in df.columns:

        df["registrations"] = pd.to_numeric(
            df["registrations"],
            errors="coerce"
        )

    # State
    if "state_name" in df.columns:

        df["state_name"] = (
            df["state_name"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

    return df


# ============================================================
# 1. LOAD ALL THREE DATASETS
# ============================================================

print("=" * 70)
print("LOADING VAHAN DATASETS")
print("=" * 70)


fuel = load_vahan(FUEL_FILE)

category = load_vahan(CATEGORY_FILE)

maker = load_vahan(MAKER_FILE)


# ============================================================
# 2. CLEAN COMMON DATA
# ============================================================

fuel = clean_common_columns(fuel)

category = clean_common_columns(category)

maker = clean_common_columns(maker)


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("CHECKING COLUMNS")
print("=" * 70)


for name, df in [
    ("Fuel", fuel),
    ("Category", category),
    ("Maker", maker)
]:

    required = [
        "date",
        "state_name",
        "registrations"
    ]

    missing = [
        col
        for col in required
        if col not in df.columns
    ]

    if missing:

        raise ValueError(
            f"{name} dataset is missing: {missing}"
        )

    print(
        f"{name}: OK"
    )


# ============================================================
# 4. REMOVE INVALID ROWS
# ============================================================

fuel = fuel.dropna(
    subset=[
        "date",
        "state_name",
        "registrations"
    ]
)

category = category.dropna(
    subset=[
        "date",
        "state_name",
        "registrations"
    ]
)

maker = maker.dropna(
    subset=[
        "date",
        "state_name",
        "registrations"
    ]
)


# ============================================================
# 5. CREATE MONTH FIELD
# ============================================================

for df in [
    fuel,
    category,
    maker
]:

    df["month"] = (
        df["date"]
        .dt.to_period("M")
        .dt.to_timestamp()
    )


# ============================================================
# 6. PROCESS FUEL TYPE
# ============================================================

print("\n" + "=" * 70)
print("PROCESSING FUEL TYPE")
print("=" * 70)


# Find fuel column

fuel_column = None

for column in [
    "fuel_type",
    "fuel",
    "fueltype",
    "type"
]:

    if column in fuel.columns:

        fuel_column = column

        break


if fuel_column is None:

    raise ValueError(
        "Fuel type column not found. "
        f"Available columns: {fuel.columns.tolist()}"
    )


print(
    "Fuel column:",
    fuel_column
)


fuel[fuel_column] = (
    fuel[fuel_column]
    .astype(str)
    .str.upper()
    .str.strip()
)


print("\nFuel types:")

print(
    fuel[fuel_column]
    .value_counts()
    .head(30)
)


# ============================================================
# 7. IDENTIFY ELECTRIC RECORDS
# ============================================================

electric_pattern = (
    r"ELECTRIC|PURE EV|BATTERY|EV"
)


ev_fuel = fuel[
    fuel[fuel_column]
    .str.contains(
        electric_pattern,
        regex=True,
        na=False
    )
].copy()


print(
    "\nElectric records:",
    len(ev_fuel)
)


if len(ev_fuel) == 0:

    raise ValueError(
        "No electric records found. "
        "Check the fuel types printed above."
    )


# ============================================================
# 8. AGGREGATE EV REGISTRATIONS
# ============================================================

ev_monthly = (
    ev_fuel
    .groupby(
        [
            "month",
            "state_name"
        ]
    )["registrations"]
    .sum()
    .reset_index()
)


ev_monthly = ev_monthly.rename(
    columns={
        "registrations":
            "ev_registrations"
    }
)


print("\nEV monthly dataset:")

print(
    ev_monthly.head()
)


# ============================================================
# 9. PROCESS VEHICLE CATEGORY
# ============================================================

print("\n" + "=" * 70)
print("PROCESSING VEHICLE CATEGORY")
print("=" * 70)


# Your uploaded category dataset uses vehicle_type

if "vehicle_type" in category.columns:

    category_column = "vehicle_type"

elif "category" in category.columns:

    category_column = "category"

elif "type" in category.columns:

    category_column = "type"

else:

    raise ValueError(
        "Vehicle category column not found. "
        f"Available: {category.columns.tolist()}"
    )


print(
    "Category column:",
    category_column
)


category[category_column] = (
    category[category_column]
    .astype(str)
    .str.upper()
    .str.strip()
)


# ============================================================
# 10. CATEGORY GROUPING
# ============================================================

def classify_category(value):

    value = str(value).upper()

    # 2 Wheeler
    if any(x in value for x in [
        "MOTOR CYCLE",
        "MOTORCYCLE",
        "SCOOTER",
        "MOPED",
        "TWO WHEELER",
        "2 WHEELER"
    ]):

        return "2W"


    # 3 Wheeler
    if any(x in value for x in [
        "THREE WHEELER",
        "3 WHEELER",
        "AUTO RICKSHAW",
        "AUTORICKSHAW"
    ]):

        return "3W"


    # Bus
    if "BUS" in value:

        return "BUS"


    # 4 Wheeler
    if any(x in value for x in [
        "MOTOR CAR",
        "CAR",
        "LIGHT MOTOR VEHICLE",
        "LIGHT PASSENGER",
        "PASSENGER"
    ]):

        return "4W"


    # Goods
    if "GOODS" in value:

        return "GOODS"


    return "OTHER"


category["vehicle_group"] = (
    category[category_column]
    .apply(classify_category)
)


print("\nVehicle groups:")

print(
    category["vehicle_group"]
    .value_counts()
)


# ============================================================
# 11. CATEGORY AGGREGATION
# ============================================================

category_monthly = (
    category
    .groupby(
        [
            "month",
            "state_name",
            "vehicle_group"
        ]
    )["registrations"]
    .sum()
    .reset_index()
)


# ============================================================
# 12. PIVOT CATEGORY
# ============================================================

category_features = (
    category_monthly
    .pivot_table(
        index=[
            "month",
            "state_name"
        ],
        columns="vehicle_group",
        values="registrations",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)


# Rename

category_features = category_features.rename(
    columns={
        "2W": "2w_registrations",
        "3W": "3w_registrations",
        "4W": "4w_registrations",
        "BUS": "bus_registrations",
        "GOODS": "goods_registrations",
        "OTHER": "other_registrations"
    }
)


# Make sure columns exist

for column in [
    "2w_registrations",
    "3w_registrations",
    "4w_registrations",
    "bus_registrations",
    "goods_registrations",
    "other_registrations"
]:

    if column not in category_features.columns:

        category_features[column] = 0


# ============================================================
# 13. PROCESS MAKER DATA
# ============================================================

print("\n" + "=" * 70)
print("PROCESSING MAKER DATA")
print("=" * 70)


if "type" in maker.columns:

    maker_column = "type"

elif "maker" in maker.columns:

    maker_column = "maker"

else:

    raise ValueError(
        "Maker column not found. "
        f"Available: {maker.columns.tolist()}"
    )


maker[maker_column] = (
    maker[maker_column]
    .astype(str)
    .str.upper()
    .str.strip()
)


# ============================================================
# 14. MAKER AGGREGATION
# ============================================================

maker_monthly = (
    maker
    .groupby(
        [
            "month",
            "state_name",
            maker_column
        ]
    )["registrations"]
    .sum()
    .reset_index()
)


# ============================================================
# 15. FIND TOP MAKERS
# ============================================================

top_makers = (
    maker_monthly
    .groupby(
        maker_column
    )["registrations"]
    .sum()
    .sort_values(
        ascending=False
    )
    .head(10)
    .index
    .tolist()
)


print("\nTop makers:")

for m in top_makers:

    print(m)


# ============================================================
# 16. FILTER TOP MAKERS
# ============================================================

top_maker_data = maker_monthly[
    maker_monthly[maker_column]
    .isin(top_makers)
].copy()


# ============================================================
# 17. PIVOT MAKER DATA
# ============================================================

maker_features = (
    top_maker_data
    .pivot_table(
        index=[
            "month",
            "state_name"
        ],
        columns=maker_column,
        values="registrations",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)


# ============================================================
# 18. CLEAN MAKER COLUMN NAMES
# ============================================================

maker_features.columns = [

    (
        "maker_"
        + re.sub(
            r"[^a-z0-9]+",
            "_",
            str(col).lower()
        ).strip("_")
    )
    if col not in [
        "month",
        "state_name"
    ]

    else col

    for col in maker_features.columns
]


# ============================================================
# 19. MERGE FUEL + CATEGORY + MAKER
# ============================================================

print("\n" + "=" * 70)
print("MERGING DATASETS")
print("=" * 70)


master = pd.merge(
    ev_monthly,
    category_features,
    on=[
        "month",
        "state_name"
    ],
    how="left"
)


master = pd.merge(
    master,
    maker_features,
    on=[
        "month",
        "state_name"
    ],
    how="left"
)


# ============================================================
# 20. FILL MISSING VALUES
# ============================================================

numeric_columns = (
    master
    .select_dtypes(
        include=np.number
    )
    .columns
)


master[numeric_columns] = (
    master[numeric_columns]
    .fillna(0)
)


# ============================================================
# 21. CREATE CATEGORY TOTALS
# ============================================================

category_cols = [
    "2w_registrations",
    "3w_registrations",
    "4w_registrations",
    "bus_registrations"
]


master["category_total"] = (
    master[category_cols]
    .sum(axis=1)
)


# ============================================================
# 22. CATEGORY SHARES
# ============================================================

master["2w_share"] = np.where(
    master["category_total"] > 0,
    master["2w_registrations"]
    /
    master["category_total"]
    * 100,
    0
)


master["3w_share"] = np.where(
    master["category_total"] > 0,
    master["3w_registrations"]
    /
    master["category_total"]
    * 100,
    0
)


master["4w_share"] = np.where(
    master["category_total"] > 0,
    master["4w_registrations"]
    /
    master["category_total"]
    * 100,
    0
)


# ============================================================
# 23. SORT DATA
# ============================================================

master = master.sort_values(
    [
        "state_name",
        "month"
    ]
).reset_index(drop=True)


# ============================================================
# 24. EV GROWTH FEATURES
# ============================================================

master["ev_growth"] = (
    master
    .groupby("state_name")["ev_registrations"]
    .pct_change()
    * 100
)


master["ev_growth_3m"] = (
    master
    .groupby("state_name")["ev_registrations"]
    .pct_change(3)
    * 100
)


master["ev_growth_12m"] = (
    master
    .groupby("state_name")["ev_registrations"]
    .pct_change(12)
    * 100
)


# ============================================================
# 25. LAG FEATURES
# ============================================================

master["ev_lag_1"] = (
    master
    .groupby("state_name")["ev_registrations"]
    .shift(1)
)


master["ev_lag_3"] = (
    master
    .groupby("state_name")["ev_registrations"]
    .shift(3)
)


master["ev_lag_6"] = (
    master
    .groupby("state_name")["ev_registrations"]
    .shift(6)
)


master["ev_lag_12"] = (
    master
    .groupby("state_name")["ev_registrations"]
    .shift(12)
)


# ============================================================
# 26. ROLLING FEATURES
# ============================================================

master["ev_rolling_3"] = (
    master
    .groupby("state_name")["ev_registrations"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(3)
        .mean()
    )
)


master["ev_rolling_6"] = (
    master
    .groupby("state_name")["ev_registrations"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(6)
        .mean()
    )
)


master["ev_rolling_12"] = (
    master
    .groupby("state_name")["ev_registrations"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(12)
        .mean()
    )
)


# ============================================================
# 27. TIME FEATURES
# ============================================================

master["year"] = (
    master["month"].dt.year
)

master["month_number"] = (
    master["month"].dt.month
)

master["quarter"] = (
    master["month"].dt.quarter
)


# ============================================================
# 28. SAVE MASTER DATASET
# ============================================================

MASTER_FILE = (
    f"{OUTPUT_DIR}/"
    "ev_master_dataset.csv"
)


master.to_csv(
    MASTER_FILE,
    index=False
)


# ============================================================
# 29. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("MASTER DATASET CREATED")
print("=" * 70)

print(
    "Shape:",
    master.shape
)

print("\nColumns:")

for column in master.columns:

    print(
        " -",
        column
    )


print("\nFirst 10 rows:")

print(
    master.head(10)
)


print("\nMissing values:")

print(
    master.isnull().sum()
)


print("\nSaved to:")

print(
    MASTER_FILE
)


# ============================================================
# 30. FINAL DATA SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    "Number of states:",
    master["state_name"].nunique()
)

print(
    "Number of months:",
    master["month"].nunique()
)

print(
    "Total EV registrations:",
    f"{master['ev_registrations'].sum():,.0f}"
)

print(
    "Date range:",
    master["month"].min(),
    "to",
    master["month"].max()
)


print("\nPipeline completed successfully.")

LOADING VAHAN DATASETS

Loading:
/content/vahan-vehicle-registrations-by-fuel-type-nmiqeb.csv
Shape: (418372, 9)
Columns:
['id', 'date', 'state_name', 'state_code', 'office_name', 'office_code', 'fuel_type', 'category', 'registrations']

Loading:
/content/vahan-vehicle-registrations-by-vehicle-category-71m55i.csv
Shape: (584267, 9)
Columns:
['id', 'date', 'state_name', 'state_code', 'office_name', 'office_code', 'vehicle_type', 'category', 'registrations']

Loading:
/content/vahan-vehicle-registrations-by-maker.csv
Shape: (2791564, 9)
Columns:
['id', 'date', 'state_name', 'state_code', 'office_name', 'office_code', 'type', 'category', 'registrations']

CHECKING COLUMNS
Fuel: OK
Category: OK
Maker: OK

PROCESSING FUEL TYPE
Fuel column: fuel_type

Fuel types:
fuel_type
PETROL                 82094
DIESEL                 81597
PETROL/HYBRID          59219
ELECTRIC(BOV)          55305
PETROL/CNG             43149
CNG ONLY               23074
NOT APPLICABLE         22206
PETROL/ETHANOL     